# CSE425: GNN-BERT Music Context - Full Colab Runner
Run every cell in order. Start with `quick`; only `final` results belong in the submitted report. Select a GPU runtime first.

In [ ]:
# Upload the project ZIP supplied with this notebook.
from google.colab import files
from pathlib import Path
import zipfile, os
uploaded = files.upload()
zip_name = next(name for name in uploaded if name.endswith('.zip'))
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall('/content')
PROJECT = Path('/content/gnn-bert-music-context')
assert PROJECT.exists(), f'Expected {PROJECT}; inspect the ZIP root.'
os.chdir(PROJECT)
print('Working in', Path.cwd())

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Enable a GPU from Runtime > Change runtime type.'
print('PyTorch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0))

In [ ]:
!python -m pip install -q -r requirements.txt
!python -m pip check

## Choose a run profile
`quick` = 1,000 tracks / 5 epochs. `final` = all tracks / up to 20 epochs. Do not use `smoke` results in the report.

In [ ]:
RUN_PROFILE = 'quick'  # change to 'final' only after quick succeeds
RUN_BONUS_TASK4 = False
print(RUN_PROFILE, RUN_BONUS_TASK4)

## Download and preprocess FMA
This downloads official metadata plus FMA-small audio. Preprocessing resumes from saved tensors when rerun.

In [ ]:
!python -m src.download_data --config config.yaml --dataset fma_small

In [ ]:
!python -m src.preprocess --config config.yaml --profile {RUN_PROFILE}
!python -m src.audit_splits --config config.yaml

## Train baselines and fusion ablations
The command trains random, CNN, BERT-only, GNN-only, early-concat, and cross-attention using identical splits.

In [ ]:
bonus_flag = '--include-bonus' if RUN_BONUS_TASK4 else ''
!python -m src.run_all --config config.yaml --profile {RUN_PROFILE} {bonus_flag}

## Inspect real results

In [ ]:
import json
from IPython.display import display, Image
metrics = json.load(open('results/metrics.json'))
for model, values in metrics.items():
    if 'macro_f1' in values:
        print(f"{model:18s} Macro-F1={values['macro_f1']:.3f} Micro-F1={values['micro_f1']:.3f} AUC-PR={values['macro_auc_pr']:.3f}")
display(Image('plots/model_comparison.png'))
display(Image('plots/fusion_tsne.png'))

## Build final submission bundle
The exporter refuses to continue if required outputs or 20 graph examples are absent.

In [ ]:
!python -m src.export_results --config config.yaml
from google.colab import files
files.download('submission_bundle.zip')

### Final manual step
Add group names/IDs and replace the bracketed interpretation sentences in `report/main.tex`. If LaTeX is unavailable in Colab, upload the report folder and plots to Overleaf's IEEE Conference template.